# 06. Разбор ошибок: почему валидация v1 завысила качество

Первая отправка (v4) дала на бенчмарке **Recall@50 = 0.926**, а на валидации v1 было **0.9585**.
Вторая (v5, + популярность объявления в train) — **0.925**. Здесь разбираемся, откуда разрыв,
и что из этого следует. Данные — артефакты итоговой версии v4 (`artifacts/`).

In [1]:
import numpy as np, pandas as pd
A = '../artifacts/'   # артефакты версии v4 (валидация v1)
qv = pd.read_parquet(A + 'queries_val.parquet'); qb = pd.read_parquet(A + 'queries_bench.parquet')
c = pd.read_parquet(A + 'corpus.parquet', columns=['item_id', 'in_bench', 'item_location_id'])
cb = c[c.in_bench == 1]; locn = cb.item_location_id.value_counts(); inb = set(cb.item_id)
# «Контрольная группа» — отложенные поиски, чьи позитивы изначально лежат в корпусе бенчмарка
qv['ctrl'] = qv.relevant.map(lambda r: all(x in inb for x in r))
oof = pd.read_parquet(A + 'stage1_oof_val.parquet')
oof['rk'] = oof.groupby('q_row').pred.rank(ascending=False, method='first')
qv['pos_rank'] = oof[oof.label == 1].groupby('q_row').rk.min().reindex(range(len(qv))).values
qv['status'] = np.where(qv.pos_rank.isna(), 'нет в пуле', np.where(qv.pos_rank <= 50, 'топ-50', 'в пуле, место > 50'))
qv['corpus_in_loc'] = qv.search_location_id.map(locn).fillna(0)
print('Контрольная группа:', qv.ctrl.sum(), 'поисков из', len(qv))
pd.crosstab(qv.status, qv.ctrl.map({True: 'позитив из корпуса бенчмарка', False: 'позитив добавлен из train'}), normalize='columns').round(3)

Контрольная группа: 418 поисков из 6000


ctrl,позитив добавлен из train,позитив из корпуса бенчмарка
status,,
"в пуле, место > 50",0.028,0.060
нет в пуле,0.009,0.010
топ-50,0.963,0.931


**Контрольная группа дала 0.931 — почти ровно результат на бенчмарке (0.926).** Остальная валидация — 0.963.
Значит, дело в объявлениях, которые мы добавляли в корпус из train.

In [2]:
# Гипотеза 1: разный состав запросов (размер города). Переучтём валидацию под распределение бенчмарка
bins = [-1, 0, 500, 2000, 5000, 1e6]; lab = ['регион/нет', '≤500', '500–2k', '2k–5k', '>5k']
qv['loc_bin'] = pd.cut(qv.corpus_in_loc, bins, labels=lab)
qb['loc_bin'] = pd.cut(qb.search_location_id.map(locn).fillna(0), bins, labels=lab)
w = qb.loc_bin.value_counts(normalize=True)
r = qv.assign(hit=qv.status == 'топ-50').groupby('loc_bin', observed=True).hit.mean()
display(pd.DataFrame({'доля в бенчмарке': w, 'Recall v4 на валидации': r}).round(3))
print('Оценка v4 с весами бенчмарка:', round((r * w).sum(), 4), '→ состав запросов разрыв не объясняет')

,доля в бенчмарке,Recall v4 на валидации
loc_bin,,
регион/нет,0.174,0.915
≤500,0.104,0.982
500–2k,0.274,0.988
2k–5k,0.226,0.983
>5k,0.221,0.914


Оценка v4 с весами бенчмарка: 0.957 → состав запросов разрыв не объясняет


In [3]:
# Гипотеза 2: корпус бенчмарка собран «вокруг» его запросов — у них больше похожих конкурентов.
# Меряем похожесть 50-го ближайшего объявления в том же городе (эмбеддинги дообученной e5-large)
E = np.load(A + 'emb_e5l_items.npy').astype(np.float32)
Qb = np.load(A + 'emb_e5l_q_bench.npy').astype(np.float32); Qv = np.load(A + 'emb_e5l_q_val.npy').astype(np.float32)
bm = c.in_bench.values == 1; loc = c.item_location_id.values
idx = {l: np.where((loc == l) & bm)[0] for l in np.unique(loc[bm])}

def cos50(Q, locs):
    out = []
    for q, l in zip(Q, locs):
        ix = idx.get(l)
        out.append(np.nan if ix is None or not (1000 <= len(ix) <= 5000) else np.partition(E[ix] @ q, -50)[-50])
    return pd.Series(out)

res = pd.Series({
    'бенчмарк': cos50(Qb, qb.search_location_id.values).median(),
    'валидация: контрольная группа': cos50(Qv[qv.ctrl.values], qv.search_location_id.values[qv.ctrl.values]).median(),
    'валидация: позитив добавлен из train': cos50(Qv[~qv.ctrl.values], qv.search_location_id.values[~qv.ctrl.values]).median(),
}, name='похожесть 50-го соседа (города 1–5 тыс. объявлений)')
res.round(3)

бенчмарк                                0.339
валидация: контрольная группа           0.326
валидация: позитив добавлен из train    0.268
Name: похожесть 50-го соседа (города 1–5 тыс. объявлений), dtype: float64

## Выводы

1. **Корпус бенчмарка собран вокруг его запросов**: при одинаковом размере города у запроса бенчмарка
   50-й ближайший конкурент заметно ближе по смыслу (0.34), чем у случайного запроса train (0.27).
   Когда мы добавляли позитив из train в чужой корпус, вокруг него почти не было похожих конкурентов — задача
   становилась легче, и валидация, и обучение ранжировщика шли на «лёгких» примерах.
2. Контрольная группа (позитивы уже в корпусе бенчмарка) устроена как бенчмарк и предсказала результат (0.931 vs 0.926).
3. Популярность объявления в train (v5) на бенчмарке не помогла (0.925): выбранные объявления бенчмарка
   редко встречаются в train. Признак исключён.

**Что сделано:** валидация v2 (`02_prepare`, `HOLDOUT_BENCH_ONLY=True`) — только поиски, у которых все выбранные
объявления уже лежат в корпусе бенчмарка (7 298 поисков); корпус валидации равен корпусу бенчмарка. Нейросети,
классификатор и гео-приор переобучены без этих поисков; ранжировщик обучается на бенчмарк-подобных примерах.

**Итог эксперимента.** Валидация v2 оценивает качество точнее (0.9435 на ней против 0.918 на бенчмарке у v6 —
разрыв меньше, чем у v1), но решение, обученное на ней, лучше не стало: v7 (3 нейросети, валидация v2) — 0.9246
на бенчмарке, v4 (валидация v1) — 0.926. Итоговой версией оставлена v4.